# ElectrideDetectives: run in Colab

Choose **Runtime → Run all**, then enter your **[OpenRouter API key](https://openrouter.ai/settings/keys)** in the masked prompt and press Enter. No code editing is needed. Use the default CPU runtime.

Setup downloads the repository and dependencies. The run uses `openai/gpt-oss-120b`, the included 200 structures, and the existing five-agent, five-round settings. OpenRouter model requests use your account's credits. Your key stays in this runtime's memory; it is not included in the notebook or result files.

Results appear below and download as a ZIP at the end. If Colab asks whether to run this GitHub notebook, review the source and choose **Run anyway** to continue.

In [ ]:
#@title 1. Set up the code and data (automatic)
import os
import subprocess
import sys
import tempfile
from pathlib import Path

print("Downloading ElectrideDetectives...")
repo_dir = Path(tempfile.mkdtemp(prefix="electride-detectives-")) / "ElectrideDetectives"
subprocess.run([
    "git", "clone", "--quiet", "--depth", "1", "--branch", "main",
    "https://github.com/sbanik2/ElectrideDetectives.git", str(repo_dir),
], check=True)
os.chdir(repo_dir)
print("Installing dependencies; the first setup may take a few minutes...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "-r", "src/requirements.txt",
], check=True)
sys.path.insert(0, str(repo_dir / "src"))
print("Setup complete. Enter your OpenRouter key below.")

In [ ]:
#@title 2. Enter your OpenRouter API key
import os, functools
from getpass import getpass
import instructor
from openai import AsyncOpenAI

MODEL_NAME = "openai/gpt-oss-120b"
_openrouter_key = os.environ.get("OPENROUTER_API_KEY", "").strip() or getpass("OpenRouter API key: ").strip()
if not _openrouter_key:
    raise ValueError("Enter an OpenRouter API key, then rerun this cell.")
client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=_openrouter_key)
del _openrouter_key
async_gen = instructor.from_openai(client, mode=instructor.Mode.JSON)
# multi-attempt retrieval to guard against err response
TEMPERATURE = 0.7                                    # exploration level for every agent call
_create = functools.partial(async_gen.chat.completions.create, temperature=TEMPERATURE, max_retries=3, max_tokens=8192)
N_TRIES = 5
async def _create_with_retry(**kwargs):
    for attempt in range(N_TRIES):
        try:
            # retries step DOWN (0.7 -> 0.55 -> 0.4 -> 0.25 -> 0.1): lower temperature helps produce valid JSON
            return await _create(**kwargs, **({"temperature": max(0.1, TEMPERATURE - 0.15 * attempt)} if attempt else {}))
        except Exception as exc:
            if attempt == N_TRIES - 1:
                raise
            print(f"LLM call failed (attempt {attempt + 1}/{N_TRIES}: {type(exc).__name__}); retrying")
async_gen.chat.completions.create = _create_with_retry

print("Selected:", MODEL_NAME, "| temperature:", TEMPERATURE)

In [ ]:
#@title 3. Run the swarm
from agent import Swarm, load_config
from evaluator import score_dataset

CONFIG = load_config("config.yaml")
CONFIG["paths"]["outputs_dir"] = "outputs/colab"
CONFIG["paths"]["arxiv_log"] = "outputs/colab/arxiv_mcp.log"

print("Preparing the paper index and classifier...")
swarm = Swarm(CONFIG, llm=async_gen, model_name=MODEL_NAME)
swarm.plot_graph()
last_round, results = await swarm.run()

## Results

The validation scores below were also used during feature selection. They are development results, not an independent test-set estimate. The exported model is refitted on all 200 structures; scoring that same dataset produces in-sample predictions.

In [ ]:
#@title 4. View the results and save the analysis
import analysis
from IPython.display import display

output_dir = Path(CONFIG["paths"]["outputs_dir"])
progress = analysis.plot_agent_progress(swarm)
progress.to_csv(output_dir / "agent_progress.csv", index=False)
display(progress)

analysis.plot_feature_map(swarm)
analysis.plot_feature_usage(swarm)
metrics = analysis.plot_final_model(swarm, results)
metrics.to_csv(output_dir / "validation_metrics.csv", index=False)
display(metrics)

scores, score_metrics = score_dataset(
    results["final_fit"], CONFIG["data"]["dataset_csv"],
    out_csv=str(output_dir / "final_model_scores.csv"),
)
display(scores.sort_values("probability", ascending=False))

In [ ]:
#@title 5. Download the run
import shutil
from google.colab import files

archive = shutil.make_archive(
    str(repo_dir.parent / "ElectrideDetectives-results"), "zip", root_dir=output_dir,
)
print("Run complete. Downloading the model, scores, logs, and analysis tables...")
files.download(archive)